In [1]:
import numpy as np
import torch
import torchvision as tv
import PIL.Image as Image
from pathlib import Path
import pandas as pd
from sklearn.preprocessing import LabelEncoder
# from torchvision.transforms import v2
import sklearn
import timm
import torch.nn as nn
from sklearn.metrics import accuracy_score
import sklearn.metrics
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
import gc
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm import tqdm
import warnings
import random
import itertools 
warnings.filterwarnings('ignore')

C:\Users\eljfe\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\scipy\__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [2]:
df = np.load(r"C:\Users\eljfe\OneDrive\Desktop\University\4 semester\Pattern M.L\datasets\development.npy")
labels = pd.read_csv(r"C:\Users\eljfe\OneDrive\Desktop\University\4 semester\Pattern M.L\datasets\development.csv")

In [3]:
useful_words = ['Licht','Radio','Ofen', 'Fernseher','Lüftung','Heizung',
                'Staubsauger','Alarm','other','an','aus']
labels = labels[labels['word'].isin(useful_words)]
encoder = LabelEncoder()
encoder.fit(labels['word'])
encoded_labels = encoder.transform(labels['word'])
labels['word_labels'] = encoded_labels
speaker_ids = labels['speaker_id'].unique()

In [4]:
len(encoder.classes_)

11

In [5]:
class CNNDataset(torch.utils.data.Dataset):
    def __init__(self, path: Path, labels_df: pd.DataFrame, data_request: int = 0):
        self.data_file = np.load(path)
        self.labels_df = labels_df
        self.data_requsted = data_request

    def __len__(self):
        return self.labels_df.shape[0]

    def __getitem__(self, idx: int):
        mel_id = int(self.labels_df.iloc[idx]['id'])
        label = self.labels_df.iloc[idx]['word_labels']
        label_torch = torch.Tensor([label])
        melspect_torch = None
        
        if not self.data_requsted:
            melspect = self.data_file[mel_id][12:76, :]
            melspect_torch = self.transform_data(melspect)

        else:
            melspect = np.concatenate((self.data_file[mel_id][:12, :], self.data_file[mel_id][76:, :]), axis=0)
            melspect_torch = self.transform_data(melspect)
            if self.data_requsted != 1:
                mel_2 = self.transform_data(self.data_file[mel_id][12:76, :])
                return melspect_torch,mel_2,label_torch
            
        return melspect_torch, label_torch

    def set_to_additional_values(self,data_request):
        self.data_requsted = data_request

    def transform_data(self,data):
        data_reshaped = data.reshape((1, data.shape[0], data.shape[1]))
        return torch.Tensor(data_reshaped)

In [6]:
class YOLODataset(torch.utils.data.Dataset):
    def __init__(self, path: Path, labels_df: pd.DataFrame): # exclude users without "other" beforehand
        self.data_file = np.load(path)                       # for clean split, since 2 out of 172 users
        self.labels_df = labels_df                           # don't have words labeled with "other"

        labels_df_group = self.labels_df.groupby(['speaker_id'])
        info_samples = []
        samples_yolo = []

        words_with_reason_l = ['Licht','Radio','Ofen','Schraube','Lüftung','Heizung','Staubsauger','Alarm']
        words_an_aus_l = ['an','aus']

        for feature_value, group in labels_df_group:
            other_words = group[group['word'] == 'other']
            all_words = group[group['word'].isin(words_with_reason_l)]
            an_aus_words = group[group['word'].isin(words_an_aus_l)]
            count_other = len(other_words)
            count_all = len(all_words)
            
            if count_other < 1: # if no words with tag "other"
                continue        # then we exclude speaker (there are 2 such speakers)

            clips = []

            for _, row in other_words.iterrows():
                clips += self.clip_data(row,2)
                clips += self.clip_data(row,1)
            random.shuffle(clips)
                
            # creating new samples
            # current pattern set to be of a view [ other + word + other + word + other]

            new_samples = []    # can be np with paddings, currently set to be a list
            new_info = []       # for padding max_len is 44 * 5 = 220, min is 90

            cur_word = 0
            cur_other = 0
            len_other = len(clips)
            len_an_aus = len(an_aus_words)
            
            while cur_word < count_all: # [ other + word1 + other + word2 + other ]

                data_samples = []

                # word1 + info about word1
                cur_word_item_1 = self.data_file[all_words.iloc[cur_word]["id"]]
                data_sample = [all_words.iloc[cur_word]["word"]] # you can add id if needed all_words.iloc[cur_word+1]["id"]

                data_sample.append(((clips[cur_other%len_other].shape[1],0),(clips[cur_other%len_other].shape[1]+44,175)))
                data_samples.append(data_sample)


                # word2 + info about word2
                cur_word_item_2 = self.data_file[an_aus_words.iloc[cur_word%len_an_aus]["id"]]
                start_word_2 = clips[cur_other%len_other].shape[1] + clips[(cur_other+1)%len_other].shape[1] + 44
                data_sample = [an_aus_words.iloc[cur_word%len_an_aus]["word"]] # you can add id if needed all_words.iloc[cur_word+1]["id"]
                data_sample.append(((start_word_2,0),(start_word_2+44,175)))
                data_samples.append(data_sample)

                cur_sample = [clips[cur_other%len_other], cur_word_item_1, clips[(cur_other+1)%len_other], cur_word_item_2, clips[(cur_other+3)%len_other]]

                new_samples.append(np.concatenate(cur_sample, axis = 1))
                new_info.append(data_samples)   

                cur_word += 1
                cur_other += 3

            info_samples += new_info
            samples_yolo += new_samples

        self.samples_yolo = samples_yolo
        self.info_samples = info_samples
        del self.data_file
        

    def __len__(self):
        return len(self.samples_yolo)

    def __getitem__(self, idx: int):
        return self.samples_yolo[idx], self.info_samples[idx]
    
    # Helpers

    def clip_data(self,sample,inersections: int = 3):
        random_numbers = sorted(random.sample(range(1, 44), inersections))
        random_numbers.append(44)
        clips_sample = []
        init_i = 0
        for rand_i in random_numbers:
            clips_sample.append(self.data_file[int(sample['id']),:,init_i:rand_i])
            init_i = rand_i
        return clips_sample

In [7]:
speaker_ids = labels['speaker_id'].unique()

train_ids, test_ids = train_test_split(speaker_ids, test_size=0.001, random_state=89)

train = labels[labels['speaker_id'].isin(train_ids)]
test = labels[labels['speaker_id'].isin(test_ids)]

print("Percentage of Training samples:",len(train)/len(labels))
print("Percentage of Testing samples:",len(test)/len(labels))
# Since it is usually same percentage as we set above with accuracy up to 1%
# There is no need to be afraid of non-balanced split since speakers were
# not perfectly distributed

Percentage of Training samples: 0.9943330251999518
Percentage of Testing samples: 0.00566697480004823


In [8]:
train_dataset = CNNDataset(r'C:\Users\eljfe\OneDrive\Desktop\University\4 semester\Pattern M.L\datasets\development.npy',train,0)
test_dataset = CNNDataset(r'C:\Users\eljfe\OneDrive\Desktop\University\4 semester\Pattern M.L\datasets\development.npy',test,0)
train_dataset.set_to_additional_values(0)
test_dataset.set_to_additional_values(0)
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

test_dataset[0][0].shape

torch.Size([1, 64, 44])

In [9]:
model = torch.hub.load('NVIDIA/DeepLearningExamples:torchhub', 'nvidia_resneXt')

n_inputs = model.fc.in_features
model.conv1 = torch.nn.Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
model.fc = nn.Sequential(
    nn.Linear(n_inputs, 512),
    nn.ReLU(),
    nn.Linear(512, 11)
)
for param in model.parameters():
    param.requires_grad = True

model

Using cache found in C:\Users\eljfe/.cache\torch\hub\NVIDIA_DeepLearningExamples_torchhub


ResNet(
  (conv1): Conv2d(1, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layers): Sequential(
    (0): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
        (bn2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(128, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Sequential(
    

In [10]:
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm

loss_fn = torch.nn.CrossEntropyLoss()

def training_step(network, optimizer, data, targets, loss_fn):
    optimizer.zero_grad()
    output = network(data)
    labels_processed = targets.flatten().long()
    loss = loss_fn(output, labels_processed)
    loss.backward()
    optimizer.step()
    return loss.item()

    
def get_metric(model, test_dataloader):
    model.eval().to('cuda')
    running_loss = 0.
    accuracy = 0.
    counter = 0
    baccuracy = 0.
    f1_scores = 0.
    for i, data in tqdm(enumerate(test_dataloader)):
        inputs, true_labels = data
        inputs = inputs.to('cuda')
        true_labels = true_labels.to('cuda')
        outputs = model(inputs)
        labels_processed = true_labels.flatten().long()
        loss = loss_fn(outputs, labels_processed)
        running_loss += loss.item()
        outputs = np.argmax(outputs.detach().cpu().numpy(), axis=1)
        labels_processed = labels_processed.detach().cpu().numpy()
        acc = accuracy_score(labels_processed, outputs)
        f1 = sklearn.metrics.f1_score(labels_processed, outputs,average="weighted")
        bacc = sklearn.metrics.balanced_accuracy_score(labels_processed, outputs)
        f1_scores += f1
        accuracy += acc
        baccuracy += bacc
        counter += 1

    print('Loss =', running_loss / counter)
    print('Accuracy = ', 100 * (accuracy / counter))
    print('F1_score = ', 100 * (f1_scores / counter))
    print('Baccuracy = ', 100 * (baccuracy / counter))



def training_loop(
        network: torch.nn.Module,
        train_dataloader,
        test_dataloader,
        num_epochs: int,
        show_progress: bool = True) -> tuple[list, list]:
    
    device = "cuda"
    device = torch.device(device)
    if not torch.cuda.is_available():
        print("CUDA IS NOT AVAILABLE")
        device = torch.device("cpu")
    losses = []

    optimizer = torch.optim.AdamW(network.parameters(), lr=0.0001)
    loss_fn = torch.nn.CrossEntropyLoss()

    for epoch in tqdm(range(num_epochs), desc="Epoch", position=0, disable= (not show_progress)):

        network.train().to('cuda')
        running_loss = 0.
        last_loss = 0.
        for i, data in tqdm(enumerate(train_dataloader), desc="Minibatch", position=1, leave=False, disable= (not show_progress)):
            inputs, targets = data
            inputs = inputs.to(device=device)
            targets = targets.to(device=device)
            loss = training_step(network, optimizer, inputs, targets,loss_fn)
            running_loss += loss
            if i % 100 == 99:
                last_loss = running_loss / 99 # loss per batch
                print('  batch {} loss: {}'.format(i + 1, last_loss))
                running_loss = 0.

        losses.append(last_loss)
        print('\n')
        get_metric(network, test_dataloader)
        print('\n\n\n')

        # scheduler.step()
            
    return network, losses

In [11]:
model,losses = training_loop(model,train_dataloader,test_dataloader,3,True)

Epoch:   0%|          | 0/3 [00:00<?, ?it/s]

  batch 100 loss: 2.049611558817854


  batch 200 loss: 0.955133076268013


  batch 300 loss: 0.5383721584021443


Minibatch: 301it [00:54,  5.15it/s]

...

Minibatch: 501it [01:31,  5.64it/s]

  batch 600 loss: 0.07941763192024835


  batch 700 loss: 0.0865755704169237


5it [00:00, 18.11it/s]
Epoch: 100%|██████████| 3/3 [07:06<00:00, 142.16s/it]

Loss = 0.079113142285496
Accuracy =  98.46153846153847
F1_score =  99.2
Baccuracy =  98.46153846153847






In [12]:
class SimpleCNN(nn.Module):
    def __init__(
            self,
            input_channels: int,
            hidden_channels: int,
            # num_hidden_layers: int,
            num_classes: int,
            use_batch_normalization: bool = True,
            kernel_size: int = 3,
            activation_function: nn.Module = nn.ReLU()): #ELU
        super().__init__()
        self.input_channels = input_channels
        self.hidden_channels = hidden_channels
        self.num_hidden_layers = len(hidden_channels)
        self.use_batch_normalization = use_batch_normalization
        self.num_classes = num_classes
        self.kernel_size = kernel_size
        self.activation_function = activation_function

        self.conv_layers = nn.ModuleList()
        
        self.conv_layers.append(nn.Conv2d(
            input_channels,
            hidden_channels[0],
            3,
            padding="same",
            padding_mode="zeros"
        ))

        for num in range(1, self.num_hidden_layers):
            self.conv_layers.append(nn.Conv2d(
                hidden_channels[num-1],
                hidden_channels[num],
                kernel_size,
                padding="same",
                padding_mode="zeros"
            ))
        if self.use_batch_normalization:
            self.batch_norm_layers = nn.ModuleList()
            for i in range(self.num_hidden_layers):
                self.batch_norm_layers.append(nn.BatchNorm2d(hidden_channels[i]))
        self.output_layer = nn.Conv2d(hidden_channels[-1], self.num_classes, kernel_size=1, stride=1)
        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(4884, 512), # 2816  if melspect 4884 if all
            nn.ReLU(),
            nn.Linear(512, 11)
        )
    
    
    def forward(self, input_images: torch.Tensor) -> torch.Tensor:

        for i in range(self.num_hidden_layers):
            input_images = self.conv_layers[i](input_images)

            if self.use_batch_normalization:
                input_images = self.batch_norm_layers[i](input_images)
            input_images = self.activation_function(input_images)
        input_images = self.output_layer(input_images)
        input_images = self.fc(input_images)

        return input_images

In [31]:
network = SimpleCNN(1, [32,64,128,256,128,64,32], 1, True, 3).to("cuda")

In [32]:
# Get data
train_dataset.set_to_additional_values(1)
test_dataset.set_to_additional_values(1)

In [33]:
# del train_dataloader
# del test_dataloader
# del network
# del clasifier
gc.collect()
torch.cuda.empty_cache()
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)
train_dataloader

In [34]:
network,losses_add = training_loop(network,train_dataloader,test_dataloader,2,True) # network

Epoch:   0%|          | 0/2 [00:00<?, ?it/s]

  batch 100 loss: 1.3881160580750667


  batch 200 loss: 0.7406397651542317


  batch 300 loss: 0.5453747556065068


  batch 400 loss: 0.3789557766432714


...

  batch 500 loss: 0.09403479881490572


  batch 600 loss: 0.10562447923226188


  batch 700 loss: 0.10214541587658753


5it [00:00, 16.87it/s]
Epoch: 100%|██████████| 2/2 [04:02<00:00, 121.01s/it]

Loss = 0.08798906859010458
Accuracy =  96.9230769230769
F1_score =  98.33333333333334
Baccuracy =  96.9230769230769






In [35]:
# del clasifier
# del melspect_c
gc.collect()
torch.cuda.empty_cache()
print(torch.cuda.get_device_name(0))
print('Memory Usage:')
print('Allocated:', round(torch.cuda.memory_allocated(0)/1024**3,1), 'GB')
print('Cached:   ', round(torch.cuda.memory_reserved(0)/1024**3,1), 'GB')

NVIDIA GeForce GTX 1660 Ti
Memory Usage:
Allocated: 0.4 GB
Cached:    0.8 GB


In [36]:
class FNN_Simple(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear((4884+input_dim), 486), # 2816  if melspect 4884 if all
            nn.ReLU(),
            nn.Linear(486, num_classes))
    
    def forward(self, x):
        return self.fc(x)

In [37]:
clasifier = FNN_Simple(n_inputs,11).to("cuda")
network.fc = nn.Flatten()
model.fc = torch.nn.Identity()

In [38]:
#Training for FNN

def training_step_class(network_1, network_2, network_clas, optimizer, data_1, data_2, targets, loss_fn):
    optimizer.zero_grad()
    output_1 = network_1(data_1)
    output_2 = network_2(data_2)
    output_1_2 = torch.cat((output_1, output_2), 1)
    output_class = network_clas(output_1_2)
    labels_processed = targets.flatten().long()
    loss = loss_fn(output_class, labels_processed)
    loss.backward()
    optimizer.step()
    return loss.item()
    
def get_metric_class(network_1: torch.nn.Module, 
                     network_2: torch.nn.Module, 
                     network_clas: torch.nn.Module, 
                     test_dataloader: torch.utils.data.dataloader.DataLoader):
    
    network_clas.eval().to('cuda')
    running_loss = 0.
    accuracy = 0.
    counter = 0
    baccuracy = 0.
    f1_scores = 0.

    print(torch.cuda.get_device_name(0))
    print('Memory Usage:')
    print('Allocated:', round(torch.cuda.memory_allocated(0)/1024**3,1), 'GB')
    print('Cached:   ', round(torch.cuda.memory_reserved(0)/1024**3,1), 'GB')

    for i, data in tqdm(enumerate(test_dataloader)):

        inputs_1, inputs_2, true_labels = data
        inputs_1 = inputs_1.to('cuda')
        inputs_2 = inputs_2.to('cuda')
        true_labels = true_labels.to('cuda')
        output_1 = network_1(inputs_1)
        output_2 = network_2(inputs_2)
        output_1_2 = torch.cat((output_1, output_2), 1)
        outputs = network_clas(output_1_2)
        labels_processed = true_labels.flatten().long()
        loss = loss_fn(outputs, labels_processed)
        running_loss += loss.item()
        outputs = np.argmax(outputs.detach().cpu().numpy(), axis=1)
        labels_processed = labels_processed.detach().cpu().numpy()
        
        acc = accuracy_score(labels_processed, outputs)
        f1 = sklearn.metrics.f1_score(labels_processed, outputs,average="weighted")
        bacc = sklearn.metrics.balanced_accuracy_score(labels_processed, outputs)

        f1_scores += f1
        accuracy += acc
        baccuracy += bacc
        counter += 1
        
    print('Loss =', running_loss / counter)
    print('Accuracy = ', 100 * (accuracy / counter))
    print('F1_score = ', 100 * (f1_scores / counter))
    print('Baccuracy = ', 100 * (baccuracy / counter))


def training_loop_class(
        network_1: torch.nn.Module,
        network_2: torch.nn.Module,
        network_clas: torch.nn.Module,
        train_dataloader: torch.utils.data.dataloader.DataLoader,
        test_dataloader: torch.utils.data.dataloader.DataLoader,
        num_epochs: int,
        show_progress: bool = True) -> tuple[list, list]:
    
    device = "cuda"
    device = torch.device(device)
    if not torch.cuda.is_available():
        print("CUDA IS NOT AVAILABLE")
        device = torch.device("cpu")
    losses = []

    optimizer = torch.optim.AdamW(network_clas.parameters(), lr=0.0001)
    loss_fn = torch.nn.CrossEntropyLoss()

    for epoch in tqdm(range(num_epochs), desc="Epoch", position=0, disable= (not show_progress)):

        network_clas.train().to('cuda')
        running_loss = 0.
        last_loss = 0.
        for i, data in tqdm(enumerate(train_dataloader), desc="Minibatch", position=1, leave=False, disable= (not show_progress)):
            inputs_1, inputs_2, targets = data
            inputs_1 = inputs_1.to(device=device)
            inputs_2 = inputs_2.to(device=device)
            targets = targets.to(device=device)
            loss = training_step_class(network_1, network_2, network_clas, optimizer, inputs_1, inputs_2, targets, loss_fn)
            running_loss += loss
            if i % 100 == 99:
                last_loss = running_loss / 99 # loss per batch
                print('  batch {} loss: {}'.format(i + 1, last_loss))
                running_loss = 0.

        losses.append(last_loss)
        print('\n')
        get_metric_class(network_1, network_2, network_clas, test_dataloader)
        print('\n\n\n')
            
    return network_1,network_2,network_clas, losses

In [39]:
train_dataset.set_to_additional_values(2)
test_dataset.set_to_additional_values(2)
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=19, shuffle=False)

In [40]:
training_loop_class(network,model,clasifier,train_dataloader,test_dataloader,2,True)

Epoch:   0%|          | 0/2 [00:00<?, ?it/s]

  batch 100 loss: 0.2699369469403543


  batch 200 loss: 0.045690747335402654


  batch 300 loss: 0.030063048486052214


  batch 400 loss: 0.04400225178276264


...

  batch 700 loss: 0.020182792875857558




NVIDIA GeForce GTX 1660 Ti
Memory Usage:
Allocated: 0.4 GB
Cached:    2.4 GB


8it [00:00,  9.04it/s]
Epoch: 100%|██████████| 2/2 [09:15<00:00, 277.79s/it]

Loss = 0.031080165192975073
Accuracy =  99.3421052631579
F1_score =  99.3859649122807
Baccuracy =  99.58333333333333






(SimpleCNN(
   (activation_function): ReLU()
   (conv_layers): ModuleList(
     (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=same)
     (1): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=same)
     (2): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=same)
     (3): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=same)
     (4): Conv2d(256, 128, kernel_size=(3, 3), stride=(1, 1), padding=same)
     (5): Conv2d(128, 64, kernel_size=(3, 3), stride=(1, 1), padding=same)
     (6): Conv2d(64, 32, kernel_size=(3, 3), stride=(1, 1), padding=same)
   )
   (batch_norm_layers): ModuleList(
     (0): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
     (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
     (2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
     (3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)

In [41]:
class Dataset_frames(torch.utils.data.Dataset):
    def __init__(self, path: Path, frame_step: int = 1, data_requsted: int = 0):
        if frame_step < 1 or not isinstance(frame_step, int):
            raise ValueError("frame step ought to be more than 0 and int")
        data_file = np.load(path)
        samples_list_wrt_frame_step = []

        if len(data_file.shape) == 2:
            data_file = data_file.reshape((-1,data_file.shape[0],data_file.shape[1]))

        length = data_file.shape[2]
        frame = 0
        while 44+frame < length:
            samples_list_wrt_frame_step.append(data_file[0,:,frame:44+frame])
            frame += frame_step
    
        self.data_file = np.array(samples_list_wrt_frame_step)
        self.data_requsted = data_requsted
        self.frame_step = frame_step
            

    def __len__(self):
        return self.data_file.shape[0]

    def __getitem__(self, idx: int):
        item = self.data_file[idx]
        melspect = item[12:76, :]
        melspect_torch = self.transform_data(melspect)

        if self.data_requsted == 1:
            mfss = np.concatenate((item[:12, :], item[76:, :]), axis=0)
            mmfss_torch = self.transform_data(mfss)
            return melspect_torch, mmfss_torch

        return melspect_torch


    def set_to_mels_and_mfss(self,flag: int = 0):
        self.data_requsted = flag

    def transform_data(self,data):
        data_reshaped = data.reshape((1, data.shape[0], data.shape[1]))
        return torch.Tensor(data_reshaped)
    

    def get_predictions(self,model: torch.nn.Module):
        model.eval().to('cuda')
        preds = []
        for _, data in enumerate(self):
            data = data.reshape((1,1, data.shape[1], data.shape[2]))
            data = data.to('cuda')
            outputs = model(data)
            outputs = np.argmax(outputs.detach().cpu().numpy(), axis=1)
            preds.append(outputs[0])

        return np.array(preds)
    

    def get_predictions_3class(self, network_1: torch.nn.Module, network_2: torch.nn.Module, network_clas: torch.nn.Module):

        melspect = self.data_file[:,12:76, :]
        melspect = melspect.reshape(melspect.shape[0],1,melspect.shape[1],melspect.shape[2])

        mfss = np.concatenate((self.data_file[:,:12, :], self.data_file[:,76:, :]), axis=1)
        mfss = mfss.reshape(mfss.shape[0],1,mfss.shape[1],mfss.shape[2])

        mfss = DataLoader(torch.Tensor(mfss), batch_size=30, shuffle=False)
        melspect = DataLoader(torch.Tensor(melspect), batch_size=30, shuffle=False)

        predictions = []
        for mfc_c, mel_c in zip(mfss,melspect):
            mfc_c = mfc_c.to(torch.device("cuda"))
            mel_c = mel_c.to(torch.device("cuda"))
            output_1 = network_1(mfc_c)
            output_2 = network_2(mel_c)

            output_1_2 = torch.cat((output_1, output_2), 1)

            outputs = network_clas(output_1_2)
            outputs = np.argmax(outputs.detach().cpu().numpy(), axis=1)
            predictions.append(outputs)

        del mfc_c
        del mel_c
        gc.collect()
        torch.cuda.empty_cache()

        predictions = np.concatenate((predictions), axis=0)

        return predictions
    
    def eval_print(self,model: torch.nn.Module = None, predictions: list = None):
        last, count_frames, start_frame = 10, 0, 0
        if predictions is None:
            predictions = self.get_predictions(model)
        for i, item in enumerate(predictions):
            if item == 10: # i.e. == other
                if last != 10:
                    print(f"Word \{encoder.classes_[last]}\ was spotted for \{count_frames}\ frames starting from \{start_frame}\ frame, i.e. \{round(1.1*start_frame*(self.frame_step/44),2)}\ second")
                    last = 10       
            else:
                if item == last:
                    count_frames += 1
                else:
                    if last != 10:
                        print(f"Word \{encoder.classes_[last]}\ was spotted for \{count_frames}\ frames starting from \{start_frame}\ frame, i.e. \{round(1.1*start_frame*(self.frame_step/44),2)}\ second")
                    count_frames,start_frame,last = 1, i, item  

    
    def predict_with_key_word_system(self,model: torch.nn.Module ,min_freq: int = 2, included_frames: int = 15, key_frames: tuple = (3,2),predictions: list = None):
        if predictions is None:
            predictions = self.get_predictions(model)
        included_frames += 1
        calls = []

        key_frames, key_frames_2 = key_frames
        
        for i, item in enumerate(predictions[key_frames:]):

            key_check = predictions[i:key_frames+i]
            unique_k, counts_k = np.unique(key_check, return_counts=True)

            if (8 in unique_k and counts_k[np.where(unique_k == 8)]>=key_frames_2) or \
                (9 in unique_k and counts_k[np.where(unique_k == 9)]>=key_frames_2):
                word_check = list(predictions[i-included_frames:key_frames+i])
                word_check = np.array(list(filter(lambda x: x not in [8,9,10], word_check))) # exclude an / aus / other

                unique, counts = np.unique(word_check, return_counts=True)

                if len(counts) != 0:
                    max_occ = counts.max()
                    if max_occ >= min_freq:
                        word = unique[np.where(counts == max_occ)][0]
                        
                        
                        key_word = unique_k[np.where(counts_k == counts_k.max())][0]
                        
                        if not any(((encoder.classes_[word] + ' ' + encoder.classes_[key_word]) == call[0]) for call in calls):
                            # meaning check:
                            timestamp = self.get_time_stamp(predictions,included_frames,word,i,key_frames)
                            calls.append((encoder.classes_[word] + ' ' + encoder.classes_[key_word],timestamp))
        return calls
    
    def get_time_stamp(self,predictions,included_frames,word,cur_step,key_frames):
        cur_step = cur_step-included_frames-key_frames
        count = 0
        for i in predictions[cur_step:]:
            if i == word:
                return (cur_step+count+2)*(self.frame_step/40)
            count += 1
            
        return (cur_step+0.3)*(self.frame_step/40)




In [42]:
# data_file = np.load(r'C:\Users\eljfe\OneDrive\Desktop\University\4 semester\Pattern M.L\datasets\scene\2_Florian_Heizung_aus.npy')
sample = Dataset_frames(r'C:\Users\eljfe\OneDrive\Desktop\University\4 semester\Pattern M.L\datasets\scene\2_Florian_Heizung_aus.npy', 2)
predictions = sample.get_predictions_3class(network,model,clasifier)
pred_np = sample.predict_with_key_word_system(model = model,min_freq = 1,included_frames = 23,key_frames = (3,2),predictions = predictions)
pred_np

[('Heizung aus', 14.3)]

In [43]:
sample.eval_print(predictions = predictions)

Word \Alarm\ was spotted for \3\ frames starting from \55\ frame, i.e. \2.75\ second
Word \Alarm\ was spotted for \1\ frames starting from \70\ frame, i.e. \3.5\ second
Word \Alarm\ was spotted for \2\ frames starting from \127\ frame, i.e. \6.35\ second
Word \Alarm\ was spotted for \1\ frames starting from \130\ frame, i.e. \6.5\ second
Word \Radio\ was spotted for \3\ frames starting from \242\ frame, i.e. \12.1\ second
Word \Heizung\ was spotted for \5\ frames starting from \284\ frame, i.e. \14.2\ second
Word \Heizung\ was spotted for \3\ frames starting from \291\ frame, i.e. \14.55\ second
Word \aus\ was spotted for \6\ frames starting from \308\ frame, i.e. \15.4\ second
Word \Alarm\ was spotted for \1\ frames starting from \385\ frame, i.e. \19.25\ second


In [44]:
sample_3 = Dataset_frames(r'C:\Users\eljfe\OneDrive\Desktop\University\4 semester\Pattern M.L\datasets\scene\5_Lukas_Staubsauger_an_Licht_aus.npy', 1)
predictions = sample_3.get_predictions_3class(network,model,clasifier)
pred_np = sample_3.predict_with_key_word_system(model = model,min_freq = 2,included_frames = 46,key_frames = (3,2),predictions = predictions)
pred_np

[('Staubsauger an', 1.35), ('Licht aus', 4.075)]

In [45]:
sample_3.eval_print(predictions = predictions)

Word \Staubsauger\ was spotted for \7\ frames starting from \52\ frame, i.e. \1.3\ second
Word \Staubsauger\ was spotted for \5\ frames starting from \60\ frame, i.e. \1.5\ second
Word \an\ was spotted for \10\ frames starting from \94\ frame, i.e. \2.35\ second
Word \Licht\ was spotted for \8\ frames starting from \161\ frame, i.e. \4.03\ second
Word \aus\ was spotted for \12\ frames starting from \198\ frame, i.e. \4.95\ second
Word \aus\ was spotted for \1\ frames starting from \223\ frame, i.e. \5.58\ second


In [46]:
dv_scenes = pd.read_csv(r"C:\Users\eljfe\OneDrive\Desktop\University\4 semester\Pattern M.L\datasets\development_scenes.csv")
dv_scenes_anots = pd.read_csv(r"C:\Users\eljfe\OneDrive\Desktop\University\4 semester\Pattern M.L\datasets\development_scene_annotations.csv")

# tt_scenes_anots = pd.read_csv(r"C:\Users\eljfe\OneDrive\Desktop\University\4 semester\Pattern M.L\datasets\test_scenes.csv")

In [47]:
test_dir = Path("test_scenes/")
test_scenes = list(test_dir.glob('*.npy'))
import os

os.path.basename(test_scenes[0])[:-4]

'00397bf2d8'

In [48]:
# # rather simple implementation of testing
# path_to_folder = r"C:\Users\eljfe\OneDrive\Desktop\University\4 semester\Pattern M.L\datasets\development_scenes"

# TP,FP,FN = 0,0,0
# str_list_validation = 'filename,command,timestamp\n'
# for i,file in tqdm(dv_scenes.iterrows()):
#     current_path = path_to_folder + '\\' + file.filename + '.npy'
#     current_sample = Dataset_frames(current_path,2)
#     predictions = current_sample.get_predictions_3class(network,model,clasifier)
#     results = current_sample.predict_with_key_word_system(model = model,min_freq = 2,included_frames = 23,key_frames = (3,2),predictions = predictions)
    
#     for item in results:
#         str_list_validation += f'{file.filename},{item[0]},{round(item[1],4)}\n'

#     # FP FN TP FP LAZY check (not including time)
#     results_no_time = [item[0] for item in results]
#     anotations = dv_scenes_anots[dv_scenes_anots['filename'] == file.filename]['command']
#     list_of_true_commands = anotations.to_list()

#     for item in results_no_time:
#         if item in list_of_true_commands:
#             TP += 1
#         else:
#             FP += 1
    
#     for item in list_of_true_commands:
#         if not item in results_no_time:
#             FN += 1

# print(f" Recall:  {TP/(TP+FN)}   |   Precision:  {TP/(TP+FP)}, TP: {TP}")
    

In [49]:
# str_list_validation

In [50]:
# f = open('csvfile_validation__with_fern.csv','w')
# f.write(str_list_validation) #Give your csv text here.
# ## Python will convert \n to os.linesep
# f.close()

In [51]:
# # rather simple implementation of testing
# import os
# path_to_folder = r"C:\Users\eljfe\OneDrive\Desktop\University\4 semester\Pattern M.L\datasets\test_scenes"

# str_list = 'filename,command,timestamp\n'
# for current_path in tqdm(test_scenes):
    
#     current_sample = Dataset_frames(current_path,2)
#     predictions = current_sample.get_predictions_3class(network,model,clasifier)
#     results = current_sample.predict_with_key_word_system(model = model,min_freq = 2,included_frames = 23,key_frames = (3,2),predictions = predictions)

#     for item in results:
#         str_list += f'{os.path.basename(current_path)[:-4]},{item[0]},{round(item[1],3)}\n'



In [52]:
# str_list

In [53]:
# f = open('csvfile_fern.csv','w')
# f.write(str_list) 
# f.close()
# csv_edit_test = pd.read_csv(r"C:\Users\eljfe\OneDrive\Desktop\University\4 semester\Pattern M.L\datasets\csvfile_fern.csv")
# csv_edit_test.timestamp = csv_edit_test.timestamp + 0.6
# csv_edit_test.to_csv('csvfile_fern.csv',index=False)

In [54]:
# f = open('csvfile_fern.csv','w')
# f.write(str_list) #Give your csv text here.
# ## Python will convert \n to os.linesep
# f.close()


In [55]:
# csv_edit = pd.read_csv(r"C:\Users\eljfe\OneDrive\Desktop\University\4 semester\Pattern M.L\datasets\csvfile_validation__with_fern.csv")
# csv_edit.timestamp = csv_edit.timestamp + 0.6
# csv_edit.to_csv('csvfile_validation__with_fern.csv',index=False)


In [56]:
diff_e_s = []
for i, file in tqdm(dv_scenes_anots.iterrows()):
    diff_e_s.append(file.end - file.start)

1190it [00:00, 13645.64it/s]


In [57]:
diff_e_s

[0.82517,
 1.7004300000000008,
 1.6258100000000013,
 1.00082,
 1.6253600000000006,
 1.47647,
 1.450610000000001,
 0.8256759999999996,
 0.8757169999999999,
 1.3014900000000003,
 1.0495199999999998,
 1.2997100000000001,
 0.8248139999999999,
 1.35041,
 1.2004100000000015,
 1.3006700000000002,
 0.9247189999999996,
 1.1496500000000012,
 1.7009299999999996,
 1.7495999999999992,
 1.0749999999999993,
 1.524989999999999,
 1.400739999999999,
 1.1001900000000013,
 1.5002600000000008,
 1.9249500000000008,
 1.6244400000000017,
 0.999547999999999,
 0.9005969999999994,
 1.0006599999999999,
 1.50089,
 1.50089,
 1.50033,
 1.4767199999999985,
 1.8009199999999996,
 1.2751799999999989,
 1.5255399999999995,
 1.325470000000001,
 1.1757199999999983,
 1.9508200000000002,
 1.3748900000000006,
 1.8259500000000024,
 2.42497,
 1.2249900000000018,
 1.3013899999999996,
 1.6247799999999994,
 1.2505100000000002,
 1.80016,
 1.5005600000000001,
 1.2758099999999999,
 1.4259099999999982,
 1.3249299999999997,
 1.199939999

In [58]:
def find_median(List): # finds the median of a sorted_list
    number_of_data = len(List)
    if number_of_data % 2 == 0:
        median = (List[(number_of_data//2)]+List[(number_of_data//2-1)])/2
    else:
        median = List[(number_of_data//2)]
    return median

middle = len(diff_e_s)//2

# lower quartile
lower_quartile = find_median(diff_e_s[:middle])

# median
median = find_median(diff_e_s)

# upper quartile
upper_quartile = find_median(diff_e_s[middle:])

In [59]:
print(f"Median: {median}, Upper Quartile: {upper_quartile}, Max: ")

Median: 0.6879275000000002, Upper Quartile: 1.2519, Max: 


In [60]:
round(upper_quartile*40)

50

In [61]:
# del df

torch.cuda.empty_cache()
gc.collect()
torch.cuda.empty_cache()
print(torch.cuda.get_device_name(0))
print('Memory Usage:')
print('Allocated:', round(torch.cuda.memory_allocated(0)/1024**3,1), 'GB')
print('Cached:   ', round(torch.cuda.memory_reserved(0)/1024**3,1), 'GB')

NVIDIA GeForce GTX 1660 Ti
Memory Usage:
Allocated: 0.4 GB
Cached:    0.9 GB


In [62]:

result_strs = ['filename,command,timestamp\n' for i in range(12)]
result_strs[0]

'filename,command,timestamp\n'

In [63]:
encoder.classes_[9]

'aus'

In [64]:
# rather simple implementation of testing
path_to_folder = r"C:\Users\eljfe\OneDrive\Desktop\University\4 semester\Pattern M.L\datasets\development_scenes"

TP_,FP_,FN_ = [0 for i in range(12)],[0 for i in range(12)],[0 for i in range(12)]

result_strs = ['filename,command,timestamp\n' for i in range(12)]
# str_list_validation_2 = 'filename,command,timestamp\n'
for i_,file in tqdm(dv_scenes.iterrows()):
    current_path = path_to_folder + '\\' + file.filename + '.npy'
    current_sample = Dataset_frames(current_path,2)

    predictions = current_sample.get_predictions_3class(network,model,clasifier)

    results_1_11 = current_sample.predict_with_key_word_system(model = model,min_freq = 1,included_frames = 28,key_frames = (1,1),predictions = predictions)
    results_1_32 = current_sample.predict_with_key_word_system(model = model,min_freq = 1,included_frames = 28,key_frames = (3,2),predictions = predictions)
    results_1_53 = current_sample.predict_with_key_word_system(model = model,min_freq = 3,included_frames = 28,key_frames = (5,3),predictions = predictions)
    results_1_43 = current_sample.predict_with_key_word_system(model = model,min_freq = 1,included_frames = 28,key_frames = (4,3),predictions = predictions)
    results_2_11 = current_sample.predict_with_key_word_system(model = model,min_freq = 2,included_frames = 28,key_frames = (1,1),predictions = predictions)
    results_2_32 = current_sample.predict_with_key_word_system(model = model,min_freq = 2,included_frames = 28,key_frames = (3,2),predictions = predictions)
    results_2_53 = current_sample.predict_with_key_word_system(model = model,min_freq = 2,included_frames = 28,key_frames = (5,3),predictions = predictions)
    results_2_43 = current_sample.predict_with_key_word_system(model = model,min_freq = 2,included_frames = 28,key_frames = (4,3),predictions = predictions)
    results_3_11 = current_sample.predict_with_key_word_system(model = model,min_freq = 3,included_frames = 28,key_frames = (1,1),predictions = predictions)
    results_3_32 = current_sample.predict_with_key_word_system(model = model,min_freq = 3,included_frames = 28,key_frames = (3,2),predictions = predictions)
    results_3_53 = current_sample.predict_with_key_word_system(model = model,min_freq = 3,included_frames = 28,key_frames = (5,3),predictions = predictions)
    results_3_43 = current_sample.predict_with_key_word_system(model = model,min_freq = 3,included_frames = 28,key_frames = (4,3),predictions = predictions)
    

    results_models = [results_1_11,results_1_32, results_1_53,results_1_43,results_2_11,results_2_32,results_2_53,results_2_43,results_3_11,
                      results_3_32,results_3_53,results_3_43]

    anotations = dv_scenes_anots[dv_scenes_anots['filename'] == file.filename]['command']
    list_of_true_commands = anotations.to_list()

    for i,result_time in enumerate(results_models):
        for item in result_time:
            result_strs[i] += f'{os.path.basename(current_path)[:-4]},{item[0]},{round(item[1]+0.6,5)}\n'
            if item[0] in list_of_true_commands:
                TP_[i] += 1
            else:
                FP_[i] += 1
        res = [item[0] for item in result_time]
        for item in list_of_true_commands:
            if not item in res:
                FN_[i] += 1

829it [19:12,  1.39s/it]


In [68]:
for TP_i, FP_i, FN_i in zip(TP_,FP_,FN_):
    print(f" Recall:  {TP_i/(TP_i+FN_i)}   |   Precision:  {TP_i/(TP_i+FP_i)}, TP: {TP_i}, FP: {FP_i}")

 Recall:  0.5579831932773109   |   Precision:  0.8426395939086294, TP: 664, FP: 124
 Recall:  0.5100840336134453   |   Precision:  0.8708751793400287, TP: 607, FP: 90
 Recall:  0.3873949579831933   |   Precision:  0.9524793388429752, TP: 461, FP: 23
 Recall:  0.45714285714285713   |   Precision:  0.8947368421052632, TP: 544, FP: 64
 Recall:  0.5159663865546219   |   Precision:  0.893740902474527, TP: 614, FP: 73
 Recall:  0.4764705882352941   |   Precision:  0.9145161290322581, TP: 567, FP: 53
 Recall:  0.4369747899159664   |   Precision:  0.9203539823008849, TP: 520, FP: 45
 Recall:  0.4327731092436975   |   Precision:  0.9262589928057554, TP: 515, FP: 41
 Recall:  0.4411764705882353   |   Precision:  0.9341637010676157, TP: 525, FP: 37
 Recall:  0.4142857142857143   |   Precision:  0.948076923076923, TP: 493, FP: 27
 Recall:  0.3873949579831933   |   Precision:  0.9524793388429752, TP: 461, FP: 23
 Recall:  0.38235294117647056   |   Precision:  0.9558823529411765, TP: 455, FP: 21


In [66]:
#results_models = [results_1_11,results_1_32, results_1_42,results_1_43,results_2_11,results_2_32,results_2_42,results_2_43,results_3_11,
             #         results_3_32,results_3_42,results_3_43]

In [69]:
for i in range(0,len(result_strs)):
    f = open(f'try_{i}.csv','w')
    f.write(result_strs[i]) #Give your csv text here.
    ## Python will convert \n to os.linesep
    f.close()